In [1]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import SimpleDirectoryReader
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.core import Settings
from llama_index.core import StorageContext
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.postgres import PGVectorStore
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from transformers import AutoTokenizer
from llama_index.core import set_global_tokenizer
from llama_index.core.text_splitter import CodeSplitter
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.packs.code_hierarchy import (
    CodeHierarchyAgentPack,
    CodeHierarchyNodeParser,
)
from pathlib import Path
from bs4 import BeautifulSoup
import psycopg 
from llama_index.core import PromptTemplate
import os
from dotenv import load_dotenv
import nest_asyncio

In [2]:
load_dotenv("/setup/on.env")
pg_user = os.getenv("POSTGRES_USER")
pg_db = os.getenv("POSTGRES_DB")
pg_pwd = os.getenv("POSTGRES_PASSWORD")

In [3]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-14B-Instruct")

Settings.embed_model = HuggingFaceEmbedding(
    model_name = "BAAI/bge-base-en-v1.5"
)

set_global_tokenizer(tokenizer.encode)

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
def drop(name):
    with psycopg.connect(
        f"host=postgres dbname={pg_db} user={pg_user} password={pg_pwd}"
    ) as conn:
        with conn.cursor() as cur:
            cur.execute(f"""
                drop table if exists {name};
                """)
            conn.commit()


drop("data_code")

In [5]:
vector_store = PGVectorStore.from_params(
    database=pg_db,
    host="postgres",
    password=pg_pwd,
    port=5432,
    user=pg_user,
    table_name="code",
    embed_dim=768,
    hnsw_kwargs={
        "hnsw_m": 14,
        "hnsw_ef_construction": 72,
        "hnsw_ef_search": 52,
        "hnsw_dist_method": "vector_cosine_ops",
    },
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [6]:
language_ext = {"python": [".py"], "lua": [".lua"], "c": [".c", ".h"]}

indexes = {}
for i in language_ext:
    documents = SimpleDirectoryReader(
        input_dir="/notebooks/data/crow-repository/",
        recursive=True,
        required_exts=language_ext[i],
        file_metadata=lambda x: {"filepath": x},
    ).load_data()
    split_nodes = CodeSplitter(
            language=i, max_chars=1024, chunk_lines=20
        ).get_nodes_from_documents(documents)
    index = VectorStoreIndex(split_nodes, storage_context=storage_context, 
                             show_progress=True, embed_model=Settings.embed_model,
                            transformations=[CodeSplitter(language=i, max_chars=1024, chunk_lines=20)])
    indexes.update({i: index})

Generating embeddings:   0%|          | 0/49 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/431 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/616 [00:00<?, ?it/s]

In [7]:
[indexes[i] for i in indexes]

In [8]:
def completion_to_prompt(completion):
   return f"<|im_start|>system\n<|im_end|>\n<|im_start|>user\n{completion}<|im_end|>\n<|im_start|>assistant\n"

def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == "system":
            prompt += f"<|im_start|>system\n{message.content}<|im_end|>\n"
        elif message.role == "user":
            prompt += f"<|im_start|>user\n{message.content}<|im_end|>\n"
        elif message.role == "assistant":
            prompt += f"<|im_start|>assistant\n{message.content}<|im_end|>\n"

    if not prompt.startswith("<|im_start|>system"):
        prompt = "<|im_start|>system\n" + prompt
        

    prompt = prompt + '<|im_start|>"You are Qwen, created by Alibaba Cloud. You are a helpful assistant. You answer user queries about the crow software library using retreived source code data from the library repository.\n'

    return prompt

llm = LlamaCPP(
    model_url="https://huggingface.co/bartowski/Qwen2.5-Coder-14B-Instruct-GGUF/resolve/main/Qwen2.5-Coder-14B-Instruct-Q6_K.gguf",
    temperature=0.1,
    max_new_tokens=1024,
    context_window=12384,
    generate_kwargs={"repeat_penalty": 1.15, "top_k": 0, "top_p": 0.5, "min_p": 0.1},
    model_kwargs={
        "n_gpu_layers": -1,
    },
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    verbose=True,
)

Settings.llm = llm

total size (MB): 12124.68


11564it [19:25,  9.92it/s]                           
ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce RTX 4060 Ti, compute capability 8.9, VMM: yes
llama_load_model_from_file: using device CUDA0 (NVIDIA GeForce RTX 4060 Ti) - 14287 MiB free
llama_model_loader: loaded meta data with 38 key-value pairs and 579 tensors from /llamaindex_cache/models/Qwen2.5-Coder-14B-Instruct-Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 Coder 14B Instruct
llama_model_loader: - kv   3:                           general.finetun

In [9]:
retriever = QueryFusionRetriever(
    [indexes[i].as_retriever() for i in indexes],
    similarity_top_k=2,
    num_queries=4,
    mode="reciprocal_rerank",
    use_async=True,
    verbose=True,
)
query_engine = RetrieverQueryEngine.from_args(retriever)
nest_asyncio.apply()

In [10]:
response = query_engine.query(
    "Using the provided context, answer the following query: Write a lua script using wtape to create loop with an endpoint that slowly changes over time."
)

llama_perf_context_print:        load time =     208.00 ms
llama_perf_context_print: prompt eval time =       0.00 ms /    81 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /    49 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =    3030.09 ms /   130 tokens


Generated queries:
1. Lua script with wtape for creating a loop with an endpoint that changes over time
2. How to use wtape in lua to create a dynamic loop endpoint
3. Lua programming: Creating a loop with changing endpoint using wtape


Llama.generate: 8 prefix-match hit, remaining 661 prompt tokens to eval
llama_perf_context_print:        load time =     208.00 ms
llama_perf_context_print: prompt eval time =       0.00 ms /   661 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /   314 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =   18964.76 ms /   975 tokens


In [11]:
print(response)

To create a loop with an endpoint that slowly changes over time using the `wtape` library in Lua, you can follow these steps:

1. Set the start of the loop.
2. Continuously update the end of the loop at a slow rate.

Here's a basic example script to achieve this:

```lua
-- Import wtape module (assuming it's available)
local wtape = require("wtape")

-- Function to set the loop start point
function set_loop_start()
    -- Set the current time as the beginning of the loop
    wtape.loop_start()
end

-- Function to update the loop end point slowly over time
function update_loop_end(slow_rate)
    local current_time = wtape.get_current_timestamp()  -- Assuming a function like this exists in wtape module
    local new_loop_end = current_time + slow_rate
    
    -- Set the updated time as the loop end, and jump to start
    wtape.loop_end(new_loop_end)
end

-- Main script execution
set_loop_start()

while true do
    update_loop_end(1)  -- Adjust this value for slower or faster change rate